# 複數(Complex Numbers)教學:Python 實作 + 應用場景

## 1. 為什麼需要複數?

在實數範圍內,`x² + 1 = 0` 無解。數學上定義虛數單位 **i**,使得 `i² = -1`,複數的形式是:

```
z = a + bj    (a 是實部,b 是虛部)
```

> Python 用 `j` 而不是 `i`(工程領域習慣),例如 `3 + 4j`。

## 2. 基本運算

```python
z1 = 3 + 4j
z2 = 1 - 2j

print(z1 + z2)          # (4+2j)
print(z1 * z2)          # (11-2j)   → (3+4j)(1-2j) = 3 - 6j + 4j - 8j² = 11 - 2j
print(z1 / z2)          # (-1+2j)
print(z1.real, z1.imag) # 3.0 4.0   -> real代表實部 , imag代表虛部
print(z1.conjugate())   # (3-4j)  共軛複數
print(abs(z1))          # 5.0     模長 = √(3²+4²)
```

## 3. 極座標與歐拉公式

複數可以看成平面上的一個點(實部 = x 軸,虛部 = y 軸),所以也能用「長度 + 角度」表示:

```
z = r·(cosθ + i·sinθ) = r·e^(iθ)    ← 歐拉公式
```

```python
import cmath, math

r, theta = cmath.polar(z1)
print(r, math.degrees(theta))     # 5.0  53.13°

z = cmath.rect(r, theta)          # 極座標轉回直角座標
print(z)                          # (3+4j)

# 歐拉恆等式 e^(iπ) + 1 = 0
print(cmath.exp(1j * math.pi))    # (-1+1.22e-16j) ≈ -1(浮點誤差)
```

## 4. 幾何意義:乘法 = 旋轉 + 縮放

這是複數最重要的直覺:**乘以 `e^(iθ)` 就是逆時針旋轉 θ 角度**。

```python
p = 2 + 1j                                  # 平面上的點 (2, 1)

# 旋轉 90 度(乘以 j)
print(p * 1j)                               # (-1+2j)

# 旋轉 30 度
rot = cmath.exp(1j * math.radians(30))
print(p * rot)                              # ≈ (1.23+1.87j)
```

不用寫旋轉矩陣,一個乘法就搞定,2D 遊戲、圖形、機器人座標轉換常這樣用。

## 5. 常見應用場景

### 5-1. 解沒有實根的方程式

```python
import numpy as np
print(np.roots([1, 2, 5]))   # x² + 2x + 5 = 0 → [-1.+2.j, -1.-2.j]
```

### 5-2. 交流電(電機、電子)

交流電的電壓、電流有**振幅 + 相位**,用複數(相量,phasor)表示後,電感、電容的計算會變成單純的複數四則運算。

```python
import cmath, math

V = 110                          # 電壓 110V(台灣家用)
f = 60                           # 頻率 60Hz
R = 10                           # 電阻 10Ω
L = 0.05                         # 電感 0.05H

w = 2 * math.pi * f
Z = R + 1j * w * L               # 阻抗 Z = R + jωL
I = V / Z                        # 電流(複數)

mag, phase = cmath.polar(I)
print(f"電流 {mag:.2f} A, 相位落後 {-math.degrees(phase):.1f}°")
# 電流 5.15 A, 相位落後 62.1°
```

電感使電流落後電壓,這個「落後」直接從複數的角度算出來。

### 5-3. 訊號處理:FFT(傅立葉轉換)

FFT 的輸出就是複數:**模長 = 該頻率的強度,角度 = 相位**。

```python
import numpy as np

fs = 1000                                   # 取樣率 1000Hz
t = np.arange(0, 1, 1/fs)
x = np.sin(2*np.pi*50*t) + 0.5*np.sin(2*np.pi*120*t)   # 50Hz + 120Hz 混合

X = np.fft.fft(x)                           # 複數陣列
freqs = np.fft.fftfreq(len(x), 1/fs)
amp = np.abs(X) / len(x) * 2                # 模長 → 振幅

top = np.argsort(amp[:len(x)//2])[-2:]
print(freqs[top], amp[top])                 # [120. 50.] [0.5 1.0]
```

用途:音訊處理、降噪、影像壓縮(JPEG)、通訊(WiFi/4G/5G 的 OFDM)、振動分析。

### 5-4. 系統穩定性(控制、時間序列)

矩陣的特徵值或多項式的根常是複數,**實部決定穩定與否,虛部決定是否震盪**:

```python
A = np.array([[0, 1], [-5, -1]])
print(np.linalg.eigvals(A))   # [-0.5+2.18j, -0.5-2.18j]
```

實部為負 → 會收斂;虛部不為零 → 會震盪。

量化中的類似例子:AR 模型的特徵多項式,所有根必須在**單位圓內**,序列才平穩。

### 5-5. 金融量化

- **選擇權定價**:Heston 模型、Carr-Madan 方法透過「特徵函數」(複數函數)搭配 FFT 求價格,比蒙地卡羅快很多。
- **週期分析**:對價格或成交量做 FFT,找出潛在週期成分(實務上雜訊很大,要小心過度擬合)。

### 5-6. 碎形(Mandelbrot Set)

```python
def mandelbrot(c, max_iter=50):
    z = 0
    for n in range(max_iter):
        z = z*z + c
        if abs(z) > 2:        # 發散
            return n
    return max_iter

print(mandelbrot(0))          # 50(不發散,在集合內)
print(mandelbrot(1+1j))       # 1(很快發散)
```

### 5-7. 量子力學

波函數本身就是複數函數,機率 = 模長平方 `|ψ|²`。量子電腦的量子位元也是用複數振幅描述。

## 6. 重點整理

| 概念 | 公式 | Python |
|---|---|---|
| 建立 | `a + bi` | `3 + 4j` |
| 模長 | `√(a²+b²)` | `abs(z)` |
| 角度 | `atan2(b, a)` | `cmath.phase(z)` |
| 共軛 | `a - bi` | `z.conjugate()` |
| 極座標 | `r·e^(iθ)` | `cmath.polar / cmath.rect` |
| 旋轉 | 乘以 `e^(iθ)` | `z * cmath.exp(1j*theta)` |

**核心直覺**:複數是「帶角度的數字」,凡是涉及**旋轉、週期、振盪、相位**的問題,用複數都會變得簡單。

如果想深入某個方向(例如 FFT 實戰、用複數畫 Mandelbrot 圖、AR 模型穩定性),可以告訴我,我再展開。

In [ ]:
z1 = complex(3,4)
print(z1)
print(z1.real)
print(z1.imag)
print(abs(z1))


(3+4j)
3.0
4.0
5.0
